# Módulo 08 — Base `ecomplexity` y gráficas del ICE municipal

Este notebook hace dos cosas:

1. **Exporta la base municipio × rama** con las variables del paquete `ecomplexity`
   (`año, clave_municipio, nom_municipio, rama, diversity, ubiquity, mcp, eci, pci, density, coi, cog, rca`)
   para el **ICE final** (el mismo del Módulo 07: UE, rama, núcleo 30/5/10, PCI espectral y proyección).
   La lógica está en `codigo/ice/export_ecomplexity.py`. En el núcleo coincide con `ecomplexity` (corr = 1).
2. **Reproduce las gráficas del notebook original** (`codigo/complejidad_economica.ipynb`) con los datos nuevos,
   organizadas por tipo de gráfica y en ciclo sobre los años donde aplica. Todas se guardan en `figuras/ice/`.

**Cómo modificar una gráfica:** cada sección tiene (a) una función `graficar_*` con el diseño y
(b) una celda de ejecución con el ciclo de años y los parámetros. Para cambiar textos, colores o tamaños,
edita la función; para cambiar años o municipios, edita la celda de ejecución. Los textos comunes
(fuente y notas) están en la celda de configuración.

**Gráficas del notebook original que no se reproducen:**
- Comparaciones de distribución PBT vs. UE y Rama vs. Clase con prueba de normalidad
  (decisión D1: la normalidad no es criterio; el diagnóstico correcto está en `outputs/ice/tabla_ue_pbt.csv`).
- Diagramas de Taylor (decisión D5: con un solo regresor no informan nada y el cruce con el PIB estaba mal).

**Requisitos:** haber corrido los Módulos 01–07 (`python -m codigo.ice.run_all`). Paquetes: numpy, pandas,
matplotlib, seaborn, networkx, pyshp (no requiere geopandas: los mapas se dibujan con `pyshp` + `matplotlib`).

## 0. Configuración

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import shapefile
from matplotlib.collections import PolyCollection

warnings.filterwarnings("ignore", category=FutureWarning)

# Raíz del repositorio: la carpeta que contiene data/ y codigo/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "codigo" / "ice").exists())
sys.path.insert(0, str(ROOT))
from codigo.ice import config as C  # noqa: E402

FIG = ROOT / "figuras" / "ice"
TAB = FIG / "tablas"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

ANIOS = C.ANIOS            # [2003, 2008, 2013, 2018, 2023]
DPI = 300
REGENERAR_BASE = False     # True para volver a calcular la base ecomplexity aunque exista

FUENTE = "Fuente: elaboración propia con datos de los Censos Económicos 2004-2024 (INEGI), años de referencia 2003-2023."
NOTA_ICE = ("El ICE se calcula con unidades económicas (UE) por rama SCIAN; el PCI se estima en un núcleo de municipios "
            "y ramas (UE totales >= 30, >= 5 ramas,\nramas presentes en >= 10 municipios) y el ICE se proyecta a todos los municipios. "
            "Estandarizado con la media y la desviación estándar del núcleo de cada año.")

REGION_CEEY = {
    "02": "Norte", "05": "Norte", "08": "Norte", "19": "Norte", "26": "Norte", "28": "Norte",
    "03": "Norte-occidente", "10": "Norte-occidente", "18": "Norte-occidente", "25": "Norte-occidente",
    "32": "Norte-occidente",
    "01": "Centro-norte", "06": "Centro-norte", "14": "Centro-norte", "16": "Centro-norte", "24": "Centro-norte",
    "09": "Centro", "11": "Centro", "13": "Centro", "15": "Centro", "17": "Centro", "21": "Centro",
    "22": "Centro", "29": "Centro",
    "04": "Sur", "07": "Sur", "12": "Sur", "20": "Sur", "23": "Sur", "27": "Sur", "30": "Sur", "31": "Sur",
}
ORDEN_REGIONES = ["Norte", "Norte-occidente", "Centro-norte", "Centro", "Sur"]


def pie(fig, texto, y=-0.02, x=0.02, fontsize=9.5):
    """Agrega fuente y notas al pie de la figura."""
    fig.text(x, y, texto, ha="left", va="top", fontsize=fontsize, linespacing=1.5)


def guardar(fig, nombre):
    """Guarda en figuras/ice/ a 300 dpi y muestra la figura en el notebook."""
    ruta = FIG / f"{nombre}.png"
    fig.savefig(ruta, dpi=DPI, bbox_inches="tight", facecolor="white")
    print(f"Guardado: {ruta.relative_to(ROOT)}")
    plt.show()
    plt.close(fig)

## 1. Exportación de la base `ecomplexity` (insumo de todas las gráficas)

In [2]:
BASES_ECO = [C.CLEAN / f"ecomplexity_rama_ue_{a}.csv.gz" for a in ANIOS]   # un archivo por año
if REGENERAR_BASE or not all(f.exists() for f in BASES_ECO):
    from codigo.ice import export_ecomplexity
    export_ecomplexity.main()

eco = pd.concat([pd.read_csv(f, dtype={"clave_municipio": str, "rama": str}) for f in BASES_ECO],
                ignore_index=True)
ramas = pd.read_csv(C.CLEAN / "catalogo_ramas.csv", dtype={"rama": str})
prox = pd.read_csv(C.CLEAN / "proximidad_ramas.csv.gz", dtype={"rama_1": str, "rama_2": str})
eco["distancia"] = 1 - eco["density"]
eco = eco.merge(ramas, on="rama", how="left")

# Una fila por municipio-año
mun = (eco.drop_duplicates(["año", "clave_municipio"])
          [["año", "clave_municipio", "nom_municipio", "eci", "coi", "diversity", "en_nucleo"]]
          .reset_index(drop=True))
mun["cve_ent"] = mun.clave_municipio.str[:2]
mun["region_ceey"] = mun.cve_ent.map(REGION_CEEY)
# Una fila por rama-año
prod = eco.drop_duplicates(["año", "rama"])[["año", "rama", "nombre_rama", "pci", "ubiquity"]].reset_index(drop=True)

print(f"Base ecomplexity: {len(eco):,} filas | municipios por año: "
      f"{mun.groupby('año').size().to_dict()} | ramas por año: {prod.groupby('año').size().to_dict()}")
eco.head()

Base ecomplexity: 3,286,998 filas | municipios por año: {2003: 2447, 2008: 2455, 2013: 2456, 2018: 2465, 2023: 2469} | ramas por año: {2003: 261, 2008: 269, 2013: 267, 2018: 269, 2023: 271}


    año clave_municipio  ... distancia                        nombre_rama
0  2003           01001  ...  0.443350                        Acuicultura
1  2003           01001  ...  0.462001                              Pesca
2  2003           01001  ...  0.441566       Extracción de petróleo y gas
3  2003           01001  ...  0.399591     Minería de minerales metálicos
4  2003           01001  ...  0.444462  Minería de minerales no metálicos

[5 rows x 16 columns]

## 2. Tablas: municipios y ramas más y menos complejos (por año)

In [3]:
def tabla_ranking_municipios(anio, n=10):
    d = mun[mun.año == anio].sort_values("eci", ascending=False)
    cols = ["clave_municipio", "nom_municipio", "eci", "diversity", "en_nucleo"]
    return d.head(n)[cols].assign(grupo="top"), d.tail(n)[cols].iloc[::-1].assign(grupo="bottom")


def tabla_ranking_ramas(anio, n=10):
    d = prod[prod.año == anio].sort_values("pci", ascending=False)
    cols = ["rama", "nombre_rama", "pci", "ubiquity"]
    return d.head(n)[cols].assign(grupo="top"), d.tail(n)[cols].iloc[::-1].assign(grupo="bottom")


tm, tr = [], []
for anio in ANIOS:
    tm += [t.assign(año=anio) for t in tabla_ranking_municipios(anio)]
    tr += [t.assign(año=anio) for t in tabla_ranking_ramas(anio)]
tm, tr = pd.concat(tm), pd.concat(tr)
tm.to_csv(TAB / "ranking_municipios_ice.csv", index=False)
tr.to_csv(TAB / "ranking_ramas_pci.csv", index=False)
print("Guardado: figuras/ice/tablas/ranking_municipios_ice.csv, ranking_ramas_pci.csv")
tm[tm.año == 2023]

Guardado: figuras/ice/tablas/ranking_municipios_ice.csv, ranking_ramas_pci.csv


      clave_municipio             nom_municipio  ...   grupo   año
10786           19006                   Apodaca  ...     top  2023
10100           09010            Álvaro Obregón  ...     top  2023
11632           22014                 Querétaro  ...     top  2023
10106           09016            Miguel Hidalgo  ...     top  2023
11844           28032                   Reynosa  ...     top  2023
10061           08037                    Juárez  ...     top  2023
10104           09014             Benito Juárez  ...     top  2023
10819           19039                 Monterrey  ...     top  2023
10799           19019    San Pedro Garza García  ...     top  2023
10590           15104       Tlalnepantla de Baz  ...     top  2023
10977           20146    San Francisco Logueche  ...  bottom  2023
11142           20311    San Pedro Jaltepetongo  ...  bottom  2023
11355           20524    Santo Domingo Yodohino  ...  bottom  2023
11399           20568         Zapotitlán Palmas  ...  bottom  

## 3. Distribución del ICE por año (histograma + KDE)

In [4]:
def graficar_distribucion_ice(anio):
    s = mun.loc[mun.año == anio, "eci"].dropna()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    sns.histplot(s, kde=True, stat="density", bins=40, color="navy", ax=ax)
    ax.axvline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(f"Distribución del Índice de Complejidad Económica a nivel municipal, México, {anio}",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Índice de Complejidad Económica (ICE)", fontweight="bold")
    ax.set_ylabel("Densidad", fontweight="bold")
    ax.text(0.98, 0.95, f"n = {len(s):,}\nasimetría = {s.skew():.2f}\ncurtosis = {s.kurtosis():.2f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(facecolor="white", edgecolor="lightgray"))
    pie(fig, f"{FUENTE}\nNotas: {NOTA_ICE}")
    return fig


def graficar_distribucion_ice_todos():
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for anio, color in zip(ANIOS, sns.color_palette("viridis", len(ANIOS))):
        sns.kdeplot(mun.loc[mun.año == anio, "eci"], ax=ax, label=str(anio), color=color, lw=2)
    ax.set_title("Distribución del ICE municipal por año censal, México, 2003-2023", fontsize=13, fontweight="bold")
    ax.set_xlabel("Índice de Complejidad Económica (ICE)", fontweight="bold")
    ax.set_ylabel("Densidad", fontweight="bold")
    ax.legend(title="Año")
    pie(fig, f"{FUENTE}\nNotas: {NOTA_ICE}\nEl ICE es relativo a cada año: las curvas comparan formas, no niveles.")
    return fig

In [5]:
for anio in ANIOS:
    guardar(graficar_distribucion_ice(anio), f"G01_distribucion_ice_{anio}")
guardar(graficar_distribucion_ice_todos(), "G01_distribucion_ice_todos_los_anios")

Guardado: figuras/ice/G01_distribucion_ice_2003.png
Guardado: figuras/ice/G01_distribucion_ice_2008.png
Guardado: figuras/ice/G01_distribucion_ice_2013.png
Guardado: figuras/ice/G01_distribucion_ice_2018.png
Guardado: figuras/ice/G01_distribucion_ice_2023.png
Guardado: figuras/ice/G01_distribucion_ice_todos_los_anios.png


## 4. Diagrama distancia–PCI de los municipios más y menos complejos (por año)

In [6]:
def graficar_distancia_pci_municipios(anio, n_mun=5, extremo="top"):
    m = mun[mun.año == anio].sort_values("eci", ascending=(extremo == "bottom")).head(n_mun)
    d = eco[(eco.año == anio) & eco.clave_municipio.isin(m.clave_municipio)]
    vmax = max(1.05, d.rca.quantile(0.95))
    norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=1, vmax=vmax)
    fig, ax = plt.subplots(figsize=(10, 7))
    sc = ax.scatter(d.distancia, d.pci, c=d.rca, cmap="coolwarm_r", norm=norm,
                    s=d.rca.clip(0.1, 10) * 40 + 20, edgecolor="black", linewidth=0.5, alpha=0.85)
    plt.colorbar(sc, ax=ax).set_label("RCA (ventaja comparativa revelada)", fontsize=11)
    texto = "mayor" if extremo == "top" else "menor"
    ax.set_title(f"Relación entre distancia tecnológica y complejidad de la industria (PCI) para los\n"
                 f"{n_mun} municipios con {texto} ICE, México, {anio}", fontsize=13, fontweight="bold", pad=15)
    ax.set_xlabel("Distancia tecnológica (1 - densidad)", fontweight="bold")
    ax.set_ylabel("PCI (complejidad de la industria)", fontweight="bold")
    ax.grid(True, ls="--", alpha=0.5)
    nombres = ", ".join(f"{r.nom_municipio} ({r.clave_municipio}, ICE {r.eci:.2f})" for r in m.itertuples())
    pie(fig, f"{FUENTE}\nNotas: cada punto es una combinación municipio-rama. El tamaño y el color reflejan el RCA.\n"
             f"Municipios: {nombres}.")
    return fig

In [7]:
for anio in ANIOS:
    guardar(graficar_distancia_pci_municipios(anio, extremo="top"), f"G02_distancia_pci_top5_municipios_{anio}")
    guardar(graficar_distancia_pci_municipios(anio, extremo="bottom"), f"G03_distancia_pci_bottom5_municipios_{anio}")

Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2003.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2003.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2008.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2008.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2013.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2013.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2018.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2018.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2023.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2023.png


## 5. Ramas más y menos complejas: PCI y distancia de los municipios (por año)

In [8]:
def etiqueta_rama(r, largo=45):
    nom = r.nombre_rama if isinstance(r.nombre_rama, str) else ""
    return f"{r.rama} {nom[:largo]}{'…' if len(nom) > largo else ''}"


def graficar_ramas_pci(anio, n_ind=5, extremo="top"):
    p = prod[prod.año == anio].sort_values("pci", ascending=(extremo == "bottom")).head(n_ind).copy()
    p["etiqueta"] = [etiqueta_rama(r) for r in p.itertuples()]
    d = eco[(eco.año == anio) & eco.rama.isin(p.rama)].merge(p[["rama", "etiqueta"]], on="rama")
    pal = "viridis" if extremo == "top" else "rocket"
    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    sns.barplot(data=p, x="pci", y="etiqueta", hue="etiqueta", palette=pal, legend=False, ax=axes[0])
    axes[0].set_title("Nivel de complejidad por rama", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("PCI (complejidad de la industria)", fontweight="bold")
    axes[0].set_ylabel("")
    for r in p.itertuples():
        axes[0].text(r.pci, r.etiqueta, f"  ubicuidad {r.ubiquity}", va="center", fontsize=8)
    sns.scatterplot(data=d, x="distancia", y="pci", hue="etiqueta", palette=pal, s=40, alpha=0.6, ax=axes[1])
    axes[1].set_title("Distancia tecnológica de cada municipio a estas ramas", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Distancia tecnológica (1 - densidad)", fontweight="bold")
    axes[1].set_ylabel("PCI (complejidad de la industria)", fontweight="bold")
    axes[1].legend(title="Rama", fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    axes[1].grid(True, ls="--", alpha=0.5)
    texto = "mayor" if extremo == "top" else "menor"
    fig.suptitle(f"Las {n_ind} ramas con {texto} complejidad de producto (PCI), México, {anio}",
                 fontsize=15, fontweight="bold", y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    pie(fig, f"{FUENTE}\nNotas: panel izquierdo, PCI de cada rama (ubicuidad = municipios del núcleo con RCA >= 1). "
             "Panel derecho, cada punto es un municipio;\nuna menor distancia indica más capacidades cercanas a la rama. "
             "El PCI de ramas con baja ubicuidad es sensible a pocos municipios (ver Módulo 04).", y=-0.12)
    return fig

In [9]:
for anio in ANIOS:
    guardar(graficar_ramas_pci(anio, extremo="top"), f"G04_top5_ramas_pci_{anio}")
    guardar(graficar_ramas_pci(anio, extremo="bottom"), f"G05_bottom5_ramas_pci_{anio}")

Guardado: figuras/ice/G04_top5_ramas_pci_2003.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2003.png
Guardado: figuras/ice/G04_top5_ramas_pci_2008.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2008.png
Guardado: figuras/ice/G04_top5_ramas_pci_2013.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2013.png
Guardado: figuras/ice/G04_top5_ramas_pci_2018.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2018.png
Guardado: figuras/ice/G04_top5_ramas_pci_2023.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2023.png


## 6. Matriz de diversificación de un municipio entre dos años

Ramas que el municipio no tenía con RCA >= 1 en el año inicial y sí en el final, contra su distancia en el año
inicial. Edita `MUNICIPIOS_DIVERSIFICACION` y `PERIODOS_DIVERSIFICACION` para agregar casos.

In [10]:
def graficar_matriz_diversificacion(clave_mun, anio_ini=2003, anio_fin=2023):
    d0 = eco[(eco.año == anio_ini) & (eco.clave_municipio == clave_mun)]
    d1 = eco[(eco.año == anio_fin) & (eco.clave_municipio == clave_mun)]
    if d0.empty or d1.empty:
        print(f"{clave_mun}: sin datos en {anio_ini} o {anio_fin}")
        return None
    nombre = d1.nom_municipio.iat[0]
    dc = d0[["rama", "pci", "distancia", "rca"]].merge(d1[["rama", "rca"]], on="rama", suffixes=("_ini", "_fin"))
    dc["nueva_ventaja"] = (dc.rca_ini < 1) & (dc.rca_fin >= 1)
    fig, ax = plt.subplots(figsize=(11, 7))
    for flag, color, lab in [(False, "lightgrey", "Mantuvo o perdió ventaja / sin ventaja"),
                             (True, "dodgerblue", "Ganó ventaja (RCA >= 1)")]:
        s = dc[dc.nueva_ventaja == flag]
        ax.scatter(s.distancia, s.pci, c=color, s=80, alpha=0.85, edgecolor="black", lw=0.5, label=lab)
    ax.axvline(dc.distancia.median(), color="firebrick", ls="--", label=f"Mediana de distancia ({anio_ini})")
    ax.set_title(f"Matriz de diversificación: adquisición de nuevas ventajas comparativas reveladas\n"
                 f"{nombre} ({clave_mun}), México, {anio_ini}-{anio_fin}", fontsize=13, fontweight="bold", pad=15)
    ax.set_xlabel(f"Distancia tecnológica inicial en {anio_ini} (1 - densidad)", fontweight="bold")
    ax.set_ylabel(f"PCI ({anio_ini})", fontweight="bold")
    ax.legend(title=f"Dinámica {anio_ini}-{anio_fin}", loc="upper left", bbox_to_anchor=(1.02, 1))
    ax.grid(True, ls="--", alpha=0.5)
    fig.tight_layout(rect=[0, 0, 0.85, 1])
    pie(fig, f"{FUENTE}\nNotas: cada punto es una rama presente en el núcleo en ambos años. Puntos azules: ramas sin RCA >= 1 en "
             f"{anio_ini} y con RCA >= 1 en {anio_fin}.\nEl PCI y la distancia son relativos a cada año.")
    return fig

In [11]:
MUNICIPIOS_DIVERSIFICACION = ["19039", "20524"]   # Monterrey; Santo Domingo Yodohino (los del notebook original)
PERIODOS_DIVERSIFICACION = [(2003, 2023)]

for cm in MUNICIPIOS_DIVERSIFICACION:
    for a0, a1 in PERIODOS_DIVERSIFICACION:
        f = graficar_matriz_diversificacion(cm, a0, a1)
        if f is not None:
            guardar(f, f"G06_diversificacion_{cm}_{a0}_{a1}")

Guardado: figuras/ice/G06_diversificacion_19039_2003_2023.png
Guardado: figuras/ice/G06_diversificacion_20524_2003_2023.png


## 7. Distribución del ICE por región CEEY (violín, por año)

In [12]:
def graficar_violin_regiones(anio):
    d = mun[mun.año == anio]
    d = pd.concat([d, d.assign(region_ceey="Nacional")])
    orden = ORDEN_REGIONES + ["Nacional"]
    colores = {r: "steelblue" for r in ORDEN_REGIONES} | {"Nacional": "lightgray"}
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.violinplot(data=d, x="region_ceey", y="eci", order=orden, hue="region_ceey", palette=colores,
                   legend=False, inner="quartile", cut=0, ax=ax)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(f"Distribución del Índice de Complejidad Económica a nivel municipal por regiones CEEY, México, {anio}",
                 fontsize=13, fontweight="bold", pad=15)
    ax.set_xlabel("Región", fontweight="bold")
    ax.set_ylabel("Índice de Complejidad Económica (ICE)", fontweight="bold")
    ax.grid(axis="y", ls="--", alpha=0.5)
    pie(fig, f"{FUENTE}\nNotas: las líneas interiores marcan los cuartiles. La línea punteada en 0 es la media del núcleo. "
             "Regiones según el CEEY.")
    return fig

In [13]:
for anio in ANIOS:
    guardar(graficar_violin_regiones(anio), f"G07_violin_regiones_{anio}")

Guardado: figuras/ice/G07_violin_regiones_2003.png
Guardado: figuras/ice/G07_violin_regiones_2008.png
Guardado: figuras/ice/G07_violin_regiones_2013.png
Guardado: figuras/ice/G07_violin_regiones_2018.png
Guardado: figuras/ice/G07_violin_regiones_2023.png


## 8. Complejidad actual (ICE) y potencial de diversificación (COI): Norte vs. Sur (por año)

In [14]:
def graficar_ice_coi(anio, regiones=("Norte", "Sur"), colores=("dodgerblue", "firebrick")):
    d = mun[(mun.año == anio)]
    fig, ax = plt.subplots(figsize=(11, 7))
    otros = d[~d.region_ceey.isin(regiones)]
    ax.scatter(otros.eci, otros.coi, c="lightgrey", s=20, alpha=0.5, label="Otras regiones")
    for reg, col in zip(regiones, colores):
        s = d[d.region_ceey == reg]
        ax.scatter(s.eci, s.coi, c=col, s=40, alpha=0.7, edgecolor="black", lw=0.3, label=reg)
    ax.axvline(0, color="black", ls="--", alpha=0.6, label="Media del núcleo (ICE = 0)")
    ax.axhline(d.coi.median(), color="gray", ls="--", alpha=0.6, label="Mediana nacional del COI")
    ax.set_title(f"Complejidad actual (ICE) y potencial de diversificación (COI) a nivel municipal,\n"
                 f"{' vs. '.join(regiones)}, México, {anio}", fontsize=13, fontweight="bold", pad=15)
    ax.set_xlabel("Índice de Complejidad Económica (ICE)", fontweight="bold")
    ax.set_ylabel("Índice de Oportunidad de Complejidad (COI)", fontweight="bold")
    ax.legend(title="Región CEEY", loc="upper left", bbox_to_anchor=(1.02, 1))
    ax.grid(True, ls="--", alpha=0.4)
    fig.tight_layout(rect=[0, 0, 0.82, 1])
    pie(fig, f"{FUENTE}\nNotas: cada punto es un municipio. El COI mide qué tan cerca está un municipio de ramas complejas que "
             "todavía no tiene;\nse estandariza con el núcleo. Los demás municipios se muestran en gris.")
    return fig

In [15]:
for anio in ANIOS:
    guardar(graficar_ice_coi(anio), f"G08_ice_coi_norte_sur_{anio}")

Guardado: figuras/ice/G08_ice_coi_norte_sur_2003.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2008.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2013.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2018.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2023.png


## 9. Matrices de transición de quintiles del ICE

Los quintiles se calculan a nivel nacional en cada año sobre los municipios presentes en ambos años.
Edita `PERIODOS_TRANSICION` para agregar pares de años.

In [16]:
ETIQ_Q = ["Q1 (más bajo)", "Q2", "Q3", "Q4", "Q5 (más alto)"]


def quintiles_movilidad(a0, a1):
    d0 = mun[mun.año == a0][["clave_municipio", "eci", "region_ceey"]]
    d1 = mun[mun.año == a1][["clave_municipio", "eci"]]
    d = d0.merge(d1, on="clave_municipio", suffixes=("_0", "_1")).dropna()
    d["Q0"] = pd.qcut(d.eci_0, 5, labels=False) + 1
    d["Q1"] = pd.qcut(d.eci_1, 5, labels=False) + 1
    return d


def heatmap_transicion(d, ax, cbar=True):
    m = pd.crosstab(d.Q0, d.Q1, normalize="index").reindex(index=range(1, 6), columns=range(1, 6), fill_value=0) * 100
    m.index, m.columns = ETIQ_Q, ETIQ_Q
    sns.heatmap(m, annot=True, fmt=".1f", cmap="Greens", vmin=0, vmax=100, linewidths=0.5, linecolor="white",
                cbar=cbar, cbar_kws={"label": "Porcentaje de municipios (%)"} if cbar else None, ax=ax)


def graficar_transicion_nacional(a0, a1):
    d = quintiles_movilidad(a0, a1)
    fig, ax = plt.subplots(figsize=(9, 7))
    heatmap_transicion(d, ax)
    ax.invert_yaxis()
    ax.set_title(f"Matriz de transición de la complejidad económica municipal, México, {a0}-{a1}",
                 fontsize=13, fontweight="bold", pad=15)
    ax.set_ylabel(f"Quintil de origen ({a0})", fontweight="bold")
    ax.set_xlabel(f"Quintil de destino ({a1})", fontweight="bold")
    pie(fig, f"{FUENTE}\nNotas: las filas suman 100%. n = {len(d):,} municipios presentes en ambos años. "
             "Quintiles nacionales del ICE de cada año.")
    return fig


def graficar_transicion_regional(a0, a1):
    d = quintiles_movilidad(a0, a1)
    regiones = ["Nacional"] + ORDEN_REGIONES
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for i, (reg, ax) in enumerate(zip(regiones, axes.flatten())):
        sub = d if reg == "Nacional" else d[d.region_ceey == reg]
        heatmap_transicion(sub, ax, cbar=(i % 3 == 2))
        ax.set_title(f"{reg} (n = {len(sub):,})", fontsize=13, fontweight="bold")
        ax.set_ylabel(f"Quintil de origen ({a0})" if i % 3 == 0 else "", fontweight="bold")
        ax.set_xlabel(f"Quintil de destino ({a1})" if i >= 3 else "", fontweight="bold")
    fig.suptitle(f"Matrices de transición de la complejidad económica municipal por región CEEY, México, {a0}-{a1}",
                 fontsize=16, fontweight="bold", y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    pie(fig, f"{FUENTE}\nNotas: las filas suman 100%. Los quintiles son nacionales para preservar la comparabilidad entre regiones.")
    return fig

In [17]:
PERIODOS_TRANSICION = [(2003, 2023), (2003, 2008), (2008, 2013), (2013, 2018), (2018, 2023)]
for a0, a1 in PERIODOS_TRANSICION:
    guardar(graficar_transicion_nacional(a0, a1), f"G09_transicion_nacional_{a0}_{a1}")
guardar(graficar_transicion_regional(2003, 2023), "G10_transicion_regional_2003_2023")

Guardado: figuras/ice/G09_transicion_nacional_2003_2023.png
Guardado: figuras/ice/G09_transicion_nacional_2003_2008.png
Guardado: figuras/ice/G09_transicion_nacional_2008_2013.png
Guardado: figuras/ice/G09_transicion_nacional_2013_2018.png
Guardado: figuras/ice/G09_transicion_nacional_2018_2023.png
Guardado: figuras/ice/G10_transicion_regional_2003_2023.png


## 10. Mapas municipales

Los polígonos se leen de `data/raw/ice/00mun_2023/00mun.shp` (Marco Geoestadístico 2023, proyección LCC de INEGI)
con `pyshp` y se dibujan con `matplotlib`. Para acelerar, `SIMPLIFICAR` toma uno de cada *k* vértices
(1 = sin simplificar).

In [18]:
SIMPLIFICAR = 3


def cargar_poligonos(k=SIMPLIFICAR):
    r = shapefile.Reader(str(C.RAW / "00mun_2023" / "00mun.shp"), encoding="latin-1")
    claves = [rec["CVEGEO"] for rec in r.records()]
    polys, dueños = [], []
    for cve, s in zip(claves, r.shapes()):
        pts = np.asarray(s.points)
        partes = list(s.parts) + [len(pts)]
        for a, b in zip(partes[:-1], partes[1:]):
            anillo = pts[a:b]
            if len(anillo) > 4 * k:
                anillo = np.vstack([anillo[::k], anillo[-1:]])
            polys.append(anillo)
            dueños.append(cve)
    return polys, np.array(dueños)


POLIGONOS, DUEÑOS = cargar_poligonos()
print(f"{len(set(DUEÑOS))} municipios, {len(POLIGONOS):,} anillos")


def dibujar_mapa(ax, valores, cmap, norm=None, colores_cat=None, gris="lightgrey"):
    """valores: Series indexada por clave municipal (numérica o categórica si colores_cat)."""
    v = valores.reindex(DUEÑOS)
    if colores_cat is not None:
        caras = [colores_cat.get(x, gris) for x in v]
    else:
        cm = plt.get_cmap(cmap)
        caras = [cm(norm(x)) if pd.notna(x) else gris for x in v]
    ax.add_collection(PolyCollection(POLIGONOS, facecolors=caras, edgecolors="black", linewidths=0.03))
    ax.autoscale_view()
    ax.set_aspect("equal")
    ax.axis("off")


def barra_color(fig, ax, cmap, norm, etiqueta, ticks=None):
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = fig.colorbar(sm, ax=ax, orientation="horizontal", shrink=0.6, pad=0.02, ticks=ticks)
    cb.set_label(etiqueta, fontsize=11)

2478 municipios, 2,908 anillos


In [19]:
def graficar_mapa_ice(anio, cmap="RdYlBu", lim=3):
    v = mun[mun.año == anio].set_index("clave_municipio").eci
    norm = mcolors.Normalize(vmin=-lim, vmax=lim)
    fig, ax = plt.subplots(figsize=(12, 9))
    dibujar_mapa(ax, v, cmap, norm)
    barra_color(fig, ax, cmap, norm, f"Índice de Complejidad Económica (ICE), escala truncada en ±{lim}")
    ax.set_title(f"Distribución espacial del Índice de Complejidad Económica a nivel municipal, México, {anio}",
                 fontsize=13, fontweight="bold", pad=15)
    pie(fig, f"{FUENTE} Polígonos: Marco Geoestadístico 2023 (INEGI).\nNotas: {NOTA_ICE}\n"
             "Gris: municipios sin dato en ese año (no existían o no tenían UE registradas).", y=0.06)
    return fig


def mov_estructural(a0, a1):
    d = quintiles_movilidad(a0, a1)
    d["delta_eci"] = d.eci_1 - d.eci_0
    d["salto_quintil"] = d.Q1 - d.Q0
    return d.set_index("clave_municipio")


def graficar_mapa_variacion(a0, a1, cmap="RdBu"):
    d = mov_estructural(a0, a1)
    lim = np.nanpercentile(np.abs(d.delta_eci), 99)
    norm = mcolors.Normalize(vmin=-lim, vmax=lim)
    fig, ax = plt.subplots(figsize=(12, 9))
    dibujar_mapa(ax, d.delta_eci, cmap, norm)
    barra_color(fig, ax, cmap, norm, f"Cambio en el ICE estandarizado ({a1} - {a0}); escala truncada en el percentil 99")
    ax.set_title(f"Cambio en la posición relativa de complejidad económica municipal, México, {a0}-{a1}",
                 fontsize=13, fontweight="bold", pad=15)
    pie(fig, f"{FUENTE}\nNotas: el ICE se estandariza en cada año, por lo que el cambio mide posición relativa, no "
             "complejidad absoluta. Azul: ganó posición; rojo: perdió posición.", y=0.06)
    return fig


TIPOLOGIAS = {
    "Liderazgo consolidado (Q4-Q5 en ambos años)": "#08519c",
    "Movilidad ascendente (subió de quintil)": "#31a354",
    "Estancamiento medio (se mantuvo en Q3)": "#d9d9d9",
    "Movilidad descendente (bajó de quintil)": "#fdae61",
    "Rezago persistente (Q1-Q2 en ambos años)": "#cb181d",
}


def clasificar_tipologia(q0, q1):
    """Mismo orden de reglas que el notebook original."""
    if q0 <= 2 and q1 <= 2:
        return "Rezago persistente (Q1-Q2 en ambos años)"
    if q1 > q0:
        return "Movilidad ascendente (subió de quintil)"
    if q1 < q0:
        return "Movilidad descendente (bajó de quintil)"
    if q0 >= 4 and q1 >= 4:
        return "Liderazgo consolidado (Q4-Q5 en ambos años)"
    return "Estancamiento medio (se mantuvo en Q3)"


def graficar_mapa_tipologias(a0, a1):
    d = mov_estructural(a0, a1)
    tip = pd.Series([clasificar_tipologia(a, b) for a, b in zip(d.Q0, d.Q1)], index=d.index)
    fig, ax = plt.subplots(figsize=(13, 9))
    dibujar_mapa(ax, tip, None, colores_cat=TIPOLOGIAS, gris="black")
    parches = [mpatches.Patch(color=c, label=f"{t} (n = {(tip == t).sum():,})") for t, c in TIPOLOGIAS.items()]
    parches.append(mpatches.Patch(color="black", label="Sin dato en alguno de los dos años"))
    ax.legend(handles=parches, loc="lower left", title=f"Dinámica de complejidad ({a0}-{a1})", fontsize=9)
    ax.set_title(f"Tipología de movilidad de la complejidad económica municipal, México, {a0}-{a1}",
                 fontsize=13, fontweight="bold", pad=15)
    pie(fig, f"{FUENTE}\nNotas: transición entre quintiles nacionales del ICE. Reglas, en orden: Q1-Q2 en ambos años = rezago; "
             "subió de quintil = ascendente;\nbajó = descendente; Q4-Q5 en ambos años sin cambio = liderazgo; resto = Q3. Negro: municipios "
             "creados después del año inicial (p. ej., San Quintín y San Felipe, B.C.).", y=0.06)
    return fig


def graficar_mapa_salto_quintil(a0, a1, cmap="RdYlGn"):
    d = mov_estructural(a0, a1)
    norm = mcolors.Normalize(vmin=-4, vmax=4)
    fig, ax = plt.subplots(figsize=(12, 9))
    dibujar_mapa(ax, d.salto_quintil, cmap, norm, gris="#f0f0f0")
    barra_color(fig, ax, cmap, norm, f"Quintiles ganados o perdidos entre {a0} y {a1}", ticks=[-4, -2, 0, 2, 4])
    ax.set_title(f"Salto de quintil del ICE municipal, México, {a0}-{a1}", fontsize=14, fontweight="bold", pad=15)
    pie(fig, f"{FUENTE}\nNotas: diferencia entre el quintil nacional del ICE en {a1} y en {a0}. 0 = mismo quintil.", y=0.06)
    return fig

In [20]:
for anio in ANIOS:
    guardar(graficar_mapa_ice(anio), f"G11_mapa_ice_{anio}")

PERIODOS_MAPA = [(2003, 2023)]
for a0, a1 in PERIODOS_MAPA:
    guardar(graficar_mapa_variacion(a0, a1), f"G12_mapa_variacion_ice_{a0}_{a1}")
    guardar(graficar_mapa_tipologias(a0, a1), f"G13_mapa_tipologias_{a0}_{a1}")
    guardar(graficar_mapa_salto_quintil(a0, a1), f"G14_mapa_salto_quintil_{a0}_{a1}")

Guardado: figuras/ice/G11_mapa_ice_2003.png
Guardado: figuras/ice/G11_mapa_ice_2008.png
Guardado: figuras/ice/G11_mapa_ice_2013.png
Guardado: figuras/ice/G11_mapa_ice_2018.png
Guardado: figuras/ice/G11_mapa_ice_2023.png
Guardado: figuras/ice/G12_mapa_variacion_ice_2003_2023.png
Guardado: figuras/ice/G13_mapa_tipologias_2003_2023.png
Guardado: figuras/ice/G14_mapa_salto_quintil_2003_2023.png


## 11. Espacio-producto de las ramas (por año)

Red de proximidad entre ramas (φ, calculada con la matriz M del núcleo; ver `export_ecomplexity.py`).
Siguiendo a Hidalgo et al. (2007): árbol de expansión máxima (MST) más los enlaces con φ >= `UMBRAL_PHI`.
El color del nodo es el PCI y el tamaño, el número de UE de la rama en el país.

*Nota sobre el notebook original:* la versión anterior agregaba enlaces con un algoritmo tipo ERGM sin semilla,
que solo podía añadir enlaces elegidos al azar entre φ ∈ (0.35, 0.65); no era reproducible ni seleccionaba los
enlaces más fuertes. Se reemplaza por la regla estándar de umbral, que es determinista.

In [21]:
UMBRAL_PHI = 0.55
SEMILLA_LAYOUT = C.SEED
ue_rama = (pd.read_csv(C.CLEAN / "saic_largo.csv.gz", dtype={"codigo": str}, usecols=["anio", "nivel", "codigo", "variable", "valor"])
             .query("nivel == 'rama' and variable == 'ue'").groupby(["anio", "codigo"]).valor.sum())


def red_espacio_producto(anio, umbral=UMBRAL_PHI):
    e = prox[(prox.año == anio) & (prox.proximidad > 0)]
    G = nx.Graph()
    G.add_weighted_edges_from(e[["rama_1", "rama_2", "proximidad"]].itertuples(index=False, name=None))
    T = nx.maximum_spanning_tree(G)
    fuertes = e[e.proximidad >= umbral]
    T.add_weighted_edges_from(fuertes[["rama_1", "rama_2", "proximidad"]].itertuples(index=False, name=None))
    return T


def graficar_espacio_producto(anio, umbral=UMBRAL_PHI):
    T = red_espacio_producto(anio, umbral)
    p = prod[prod.año == anio].set_index("rama")
    pos = nx.spring_layout(T, weight="weight", seed=SEMILLA_LAYOUT, k=0.18, iterations=300)
    nodos = list(T.nodes())
    pci = p.pci.reindex(nodos).to_numpy()
    tam = np.sqrt(ue_rama.loc[anio].reindex(nodos).fillna(1).to_numpy())
    tam = 15 + 285 * (tam - tam.min()) / (tam.max() - tam.min())
    fig, ax = plt.subplots(figsize=(13, 11))
    nx.draw_networkx_edges(T, pos, ax=ax, alpha=0.25, width=0.6, edge_color="gray")
    norm = mcolors.TwoSlopeNorm(vmin=np.nanmin(pci), vcenter=0, vmax=np.nanmax(pci))
    sc = nx.draw_networkx_nodes(T, pos, nodelist=nodos, node_color=pci, cmap="RdYlBu_r", vmin=norm.vmin,
                                vmax=norm.vmax, node_size=tam, edgecolors="black", linewidths=0.3, ax=ax)
    top = p.pci.reindex(nodos).nlargest(8).index
    nx.draw_networkx_labels(T, pos, labels={r: r for r in top}, font_size=7, ax=ax)
    plt.colorbar(sc, ax=ax, shrink=0.6).set_label("PCI de la rama")
    ax.set_title(f"Espacio-producto de las ramas económicas, México, {anio}", fontsize=14, fontweight="bold")
    ax.axis("off")
    pie(fig, f"{FUENTE}\nNotas: nodos = ramas del núcleo; enlaces = árbol de expansión máxima de la proximidad φ más los enlaces "
             f"con φ >= {umbral}\n({T.number_of_nodes()} nodos, {T.number_of_edges()} enlaces). Tamaño proporcional a la raíz "
             "de las UE de la rama. Se etiquetan las 8 ramas de mayor PCI.", y=0.04)
    return fig

In [22]:
for anio in ANIOS:
    guardar(graficar_espacio_producto(anio), f"G15_espacio_producto_{anio}")

Guardado: figuras/ice/G15_espacio_producto_2003.png
Guardado: figuras/ice/G15_espacio_producto_2008.png
Guardado: figuras/ice/G15_espacio_producto_2013.png
Guardado: figuras/ice/G15_espacio_producto_2018.png
Guardado: figuras/ice/G15_espacio_producto_2023.png


## 12. Figuras del Apéndice Metodológico (`figuras/ice/anexo/`)

Dos figuras de diagnóstico que usa `docs/anexo_ice.docx`: censura del PBT por tamaño del municipio y
condicionamiento espectral UE vs. PBT. Leen `outputs/ice/03_censura_pbt.csv` y `outputs/ice/tabla_ue_pbt.csv`
(Módulo 03).

In [23]:
FIG_ANEXO = FIG / "anexo"
FIG_ANEXO.mkdir(exist_ok=True)


def graficar_censura_pbt():
    c = pd.read_csv(C.OUT / "03_censura_pbt.csv")
    q = c[c.desglose == "quintil_ue_mun"].pivot(index="quintil_ue_mun", columns="anio", values="pct_suprimida")
    fig, ax = plt.subplots(figsize=(10, 5.5))
    w, x = 0.15, np.arange(5)
    cols = plt.cm.viridis(np.linspace(0.15, 0.85, len(q.columns)))
    for i, a in enumerate(q.columns):
        ax.bar(x + (i - 2) * w, q[a], w, label=str(a), color=cols[i])
    ax.set_xticks(x)
    ax.set_xticklabels(["Q1\n(más chicos)", "Q2", "Q3", "Q4", "Q5\n(más grandes)"])
    ax.set_ylabel("% de celdas rama×municipio con PBT suprimido", fontweight="bold")
    ax.set_xlabel("Quintil de unidades económicas totales del municipio", fontweight="bold")
    ax.set_ylim(0, 100)
    ax.legend(title="Año", ncol=5, loc="upper right")
    ax.grid(axis="y", ls="--", alpha=0.5)
    ax.set_title("Censura del PBT por confidencialidad según tamaño del municipio, México, 2003-2023",
                 fontsize=13, fontweight="bold")
    pie(fig, f"{FUENTE}\nNotas: proporción de celdas con UE > 0 cuyo PBT aparece vacío en SAIC (dato omitido por confidencialidad).")
    return fig


def graficar_espectro_ue_pbt():
    t = pd.read_csv(C.OUT / "tabla_ue_pbt.csv")
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    for esp, col, mk in [("UE (núcleo)", "navy", "o"), ("PBT (mismo núcleo)", "firebrick", "s")]:
        s = t[t.especificacion == esp].set_index("anio")
        axes[0].plot(s.index, s.lambda2, marker=mk, color=col, label=f"{esp}: λ2")
        axes[0].plot(s.index, s.lambda3, marker=mk, color=col, ls="--", alpha=0.7, label=f"{esp}: λ3")
        axes[1].plot(s.index, s.empates_pct, marker=mk, color=col, label=esp)
    axes[0].set_title("Segundo y tercer eigenvalor de la matriz M̃", fontweight="bold")
    axes[0].set_xlabel("Año"); axes[0].set_ylabel("Eigenvalor"); axes[0].set_ylim(0, 1.05)
    axes[0].legend(fontsize=8); axes[0].grid(ls="--", alpha=0.5)
    axes[1].set_title("Municipios con ICE empatado (%)", fontweight="bold")
    axes[1].set_xlabel("Año"); axes[1].set_ylabel("%"); axes[1].legend(fontsize=9); axes[1].grid(ls="--", alpha=0.5)
    fig.suptitle("Condicionamiento de la matriz de complejidad: UE frente a PBT, México, 2003-2023",
                 fontsize=13, fontweight="bold", y=1.0)
    fig.tight_layout()
    pie(fig, f"{FUENTE}\nNotas: ambos índices se calculan sobre el mismo núcleo de municipios y ramas; en PBT las celdas "
             "suprimidas cuentan como cero. λ2 = 1 indica una matriz partida en bloques\ndesconectados (2023); "
             "λ2 ≈ λ3 indica que el eigenvector no está identificado (2018).")
    return fig


guardar(graficar_censura_pbt(), "anexo/A01_censura_pbt_por_quintil")
guardar(graficar_espectro_ue_pbt(), "anexo/A02_espectro_ue_vs_pbt")

Guardado: figuras/ice/anexo/A01_censura_pbt_por_quintil.png
Guardado: figuras/ice/anexo/A02_espectro_ue_vs_pbt.png


## 13. Inventario de figuras generadas

In [24]:
figs = sorted(str(p.relative_to(FIG)) for p in FIG.rglob("*.png"))
print(f"{len(figs)} figuras en {FIG.relative_to(ROOT)}")
pd.Series(figs).str.extract(r"^(G\d+)")[0].value_counts().sort_index()

59 figuras en figuras/ice


0
G01    6
G02    5
G03    5
G04    5
G05    5
G06    2
G07    5
G08    5
G09    5
G10    1
G11    5
G12    1
G13    1
G14    1
G15    5
Name: count, dtype: int64